# 02 · Ingest Master Meter 1

**PRT661 Assessment 2 · Group 4 · DKASC Alice Springs**

Ingests `Datasets/reference/96-Site_DKA-MasterMeter1.csv` into an analysis ready store.

This file, rather than the 19 yearly extracts, is the project's primary dataset. It is the
series the published benchmark reports on, it carries the meteorological covariates the yearly
extracts lack entirely, and it needs no schema reconciliation.

Steps: load, enforce timestamp integrity against a complete 5 minute grid, measure data quality
before any cleaning, screen for physically impossible values, aggregate to hourly, and write
Parquet.

Emits `Outputs/data_quality_mastermeter1.md`, cited in the Assessment 2 report.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import time

REPO = Path("/Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting")
REF  = REPO/"Datasets"/"reference"
OUT  = REPO/"Outputs"
PROC = REPO/"Datasets"/"processed"
PROC.mkdir(parents=True, exist_ok=True)

SRC = REF/"96-Site_DKA-MasterMeter1.csv"

t0 = time.time()
df = pd.read_csv(SRC, parse_dates=["timestamp"], low_memory=False)
print(f"loaded {len(df):,} rows x {df.shape[1]} cols in {time.time()-t0:.1f}s")
print(f"memory: {df.memory_usage(deep=True).sum()/1e6:.0f} MB")
print()
print(df.dtypes.to_string())

loaded 1,767,903 rows x 17 cols in 3.4s
memory: 240 MB

timestamp                           datetime64[ns]
Active_Energy_Delivered_Received           float64
Current_Phase_Average                      float64
Active_Power                               float64
Power_Factor_Signed                        float64
Average_Voltage_Line_to_Neutral            float64
Frequency                                  float64
THD_Voltage_Average                        float64
Wind_Speed                                 float64
Weather_Temperature_Celsius                float64
Weather_Relative_Humidity                  float64
Global_Horizontal_Radiation                float64
Diffuse_Horizontal_Radiation               float64
Wind_Direction                             float64
Weather_Daily_Rainfall                     float64
Radiation_Global_Tilted                    float64
Radiation_Diffuse_Tilted                   float64


## 1 · Timestamp integrity and the complete grid

Reindexing onto a complete 5 minute grid converts implicit gaps, where a reading simply does not
appear, into explicit missing values. Without this step missingness is invisible and any lag
feature silently spans an unknown interval.

In [2]:
df = df.sort_values("timestamp").reset_index(drop=True)

n_raw = len(df)
n_dup = int(df["timestamp"].duplicated().sum())
t_min, t_max = df["timestamp"].min(), df["timestamp"].max()

df = df.drop_duplicates(subset="timestamp", keep="first")

grid = pd.date_range(t_min, t_max, freq="5min")
present  = df["timestamp"].isin(grid).sum()
off_grid = len(df) - present

df = df.set_index("timestamp").reindex(grid)
df.index.name = "timestamp"

n_gridded  = len(df)
n_inserted = n_gridded - (n_raw - n_dup)

print(f"raw rows                 : {n_raw:,}")
print(f"duplicate timestamps     : {n_dup:,}")
print(f"rows not on the 5min grid: {off_grid:,}")
print(f"coverage                 : {t_min}  to  {t_max}")
print(f"complete 5min grid       : {len(grid):,} slots")
print(f"rows after reindex       : {n_gridded:,}")
print(f"empty slots inserted     : {n_inserted:,}  ({100*n_inserted/n_gridded:.2f}% of the grid)")

raw rows                 : 1,767,903
duplicate timestamps     : 0
rows not on the 5min grid: 0
coverage                 : 2008-09-12 05:55:00  to  2025-08-23 05:00:00
complete 5min grid       : 1,782,422 slots
rows after reindex       : 1,782,422
empty slots inserted     : 14,519  (0.81% of the grid)


## 2 · Measured data quality, before any cleaning

Baseline figures. Every cleaning decision downstream is justified against this table, and the before and after pair is what the marking rubric asks for.

In [3]:
rows = []
for c in df.columns:
    s = pd.to_numeric(df[c], errors="coerce")
    rows.append({
        "column":    c,
        "missing_%": round(100*s.isna().mean(), 2),
        "min":       round(s.min(), 3) if s.notna().any() else None,
        "max":       round(s.max(), 3) if s.notna().any() else None,
        "mean":      round(s.mean(), 3) if s.notna().any() else None,
        "zeros_%":   round(100*(s == 0).mean(), 2),
    })
quality_before = pd.DataFrame(rows)
pd.set_option("display.width", 170, "display.max_colwidth", 40)
print(quality_before.to_string(index=False))

                          column  missing_%        min        max       mean  zeros_%
Active_Energy_Delivered_Received       2.34    -68.166 999999.000 466144.560     0.00
           Current_Phase_Average      41.67      0.000    366.075     74.208     0.02
                    Active_Power       2.34     -1.861    241.026     44.731     0.06
             Power_Factor_Signed      41.70   -100.000    100.000      8.614     0.00
 Average_Voltage_Line_to_Neutral       2.31      0.000    269.254    243.104     0.06
                       Frequency       2.35      0.000     53.000     49.986     0.03
             THD_Voltage_Average      41.67      0.000     11.984      1.353     0.02
                      Wind_Speed      52.77  -1742.131     54.389      2.509     0.97
     Weather_Temperature_Celsius       2.96    -39.988     61.924     21.155     0.29
       Weather_Relative_Humidity       2.96      0.000    131.158     38.805     0.29
     Global_Horizontal_Radiation       2.96   -985.690

## 3 · Physical validity screen

Statistical outlier detection cannot distinguish a genuine extreme from an impossible one. These rules encode domain constraints: irradiance is never negative, relative humidity is bounded, grid frequency sits near 50 Hz.

In [4]:
TARGET = "Active_Power"

checks = {
    "Global_Horizontal_Radiation":  ("negative irradiance",      lambda s: s < 0),
    "Diffuse_Horizontal_Radiation": ("negative irradiance",      lambda s: s < 0),
    "Radiation_Global_Tilted":      ("negative irradiance",      lambda s: s < 0),
    "Radiation_Diffuse_Tilted":     ("negative irradiance",      lambda s: s < 0),
    "Weather_Relative_Humidity":    ("outside 0 to 100",         lambda s: (s < 0) | (s > 100)),
    "Weather_Temperature_Celsius":  ("outside -10 to 55 C",      lambda s: (s < -10) | (s > 55)),
    "Wind_Speed":                   ("negative or above 50 m/s", lambda s: (s < 0) | (s > 50)),
    "Frequency":                    ("outside 45 to 55 Hz",      lambda s: (s < 45) | (s > 55)),
}

print(f"{'column':<32} {'rule':<26} {'violations':>12} {'%':>8}")
print("-" * 82)
violations = {}
for c, (label, rule) in checks.items():
    if c not in df.columns:
        continue
    s = pd.to_numeric(df[c], errors="coerce")
    mask = rule(s) & s.notna()
    violations[c] = int(mask.sum())
    print(f"{c:<32} {label:<26} {mask.sum():>12,} {100*mask.mean():>7.3f}%")

s = pd.to_numeric(df[TARGET], errors="coerce")
print()
print(f"{TARGET}: min {s.min():.3f}  max {s.max():.3f}  mean {s.mean():.3f}")
print(f"   negative values : {int((s < 0).sum()):,}  ({100*(s < 0).mean():.2f}%)")
print(f"   night-time zeros: {int((s == 0).sum()):,}")

column                           rule                         violations        %
----------------------------------------------------------------------------------
Global_Horizontal_Radiation      negative irradiance                   1   0.000%
Diffuse_Horizontal_Radiation     negative irradiance                   0   0.000%
Radiation_Global_Tilted          negative irradiance                   3   0.000%
Radiation_Diffuse_Tilted         negative irradiance                   1   0.000%
Weather_Relative_Humidity        outside 0 to 100                 14,546   0.816%
Weather_Temperature_Celsius      outside -10 to 55 C               3,049   0.171%
Wind_Speed                       negative or above 50 m/s             18   0.001%
Frequency                        outside 45 to 55 Hz                 494   0.028%

Active_Power: min -1.861  max 241.026  mean 44.731
   negative values : 900,465  (50.52%)
   night-time zeros: 1,000


## 4 · Hourly aggregation

The stated forecast horizons are 1 hour and 24 hours ahead, so hourly is the natural modelling
resolution. It also reduces 1.77M rows to roughly 148k, which makes walk forward validation
tractable.

The cumulative energy counter is differenced first and then summed, never averaged. Negative
differences indicate a counter reset and become missing rather than being clipped, which would
fabricate zero energy for those intervals.

In [5]:
WEATHER = ["Wind_Speed", "Weather_Temperature_Celsius", "Weather_Relative_Humidity",
           "Global_Horizontal_Radiation", "Diffuse_Horizontal_Radiation", "Wind_Direction",
           "Weather_Daily_Rainfall", "Radiation_Global_Tilted", "Radiation_Diffuse_Tilted"]
ELECTRICAL = ["Active_Power", "Current_Phase_Average", "Power_Factor_Signed",
              "Average_Voltage_Line_to_Neutral", "Frequency", "THD_Voltage_Average"]
COUNTER = "Active_Energy_Delivered_Received"

num = df.apply(pd.to_numeric, errors="coerce")

delta = num[COUNTER].diff()
delta[delta < 0] = np.nan
num["interval_energy"] = delta

agg = {c: "mean" for c in WEATHER + ELECTRICAL if c in num.columns}
agg["interval_energy"] = "sum"

hourly = num.resample("1h").agg(agg)
hourly["n_obs"] = num[TARGET].resample("1h").count()

print(f"5min rows  : {len(num):,}")
print(f"hourly rows: {len(hourly):,}")
print(f"expected   : {int((num.index.max() - num.index.min()).total_seconds()//3600)+1:,}")
print()
print(f"hours with all 12 readings : {int((hourly['n_obs'] == 12).sum()):,}")
print(f"hours with zero readings   : {int((hourly['n_obs'] == 0).sum()):,}")
print()
cols = [TARGET, "Global_Horizontal_Radiation", "Weather_Temperature_Celsius", "n_obs"]
print(hourly[cols].describe().round(3).to_string())

5min rows  : 1,782,422
hourly rows: 148,537
expected   : 148,536

hours with all 12 readings : 136,246
hours with zero readings   : 1,223

       Active_Power  Global_Horizontal_Radiation  Weather_Temperature_Celsius       n_obs
count    147314.000                   146288.000                   146288.000  148537.000
mean         44.765                      262.070                       21.173      11.719
std          61.323                      355.025                        9.864       1.315
min          -1.787                        0.000                      -39.988       0.000
25%          -0.476                        2.851                       14.504      12.000
50%           0.099                       11.820                       21.700      12.000
75%          89.600                      527.290                       28.302      12.000
max         209.159                     1373.262                       45.229      12.000


## 5 · Write Parquet and the data quality report

Parquet is columnar and compressed, so downstream notebooks load only the columns they need. The files stay out of version control; the quality report does not.

In [6]:
p5, p1 = PROC/"mastermeter1_5min.parquet", PROC/"mastermeter1_hourly.parquet"
num.to_parquet(p5, compression="snappy")
hourly.to_parquet(p1, compression="snappy")
print(f"{p5.name:<34} {p5.stat().st_size/1e6:>8.1f} MB   ({len(num):,} rows)")
print(f"{p1.name:<34} {p1.stat().st_size/1e6:>8.1f} MB   ({len(hourly):,} rows)")

lines = [
    "# Data Quality Report: Master Meter 1",
    "",
    "Generated by `notebooks/02_ingest_mastermeter1.ipynb`.",
    f"Source: `Datasets/reference/{SRC.name}` ({SRC.stat().st_size/1e6:.0f} MB).",
    "",
    "## Ingestion",
    "",
    "| Measure | Value |",
    "|---|---|",
    f"| Rows as supplied | {n_raw:,} |",
    f"| Duplicate timestamps removed | {n_dup:,} |",
    f"| Rows off the 5 minute grid | {off_grid:,} |",
    f"| Coverage | {t_min} to {t_max} |",
    f"| Complete 5 minute grid | {len(grid):,} slots |",
    f"| Empty slots inserted by reindexing | {n_inserted:,} ({100*n_inserted/n_gridded:.2f}%) |",
    f"| Hourly rows after aggregation | {len(hourly):,} |",
    "",
    "## Physical validity violations",
    "",
    "| Column | Violations |",
    "|---|---|",
]
lines += [f"| `{c}` | {v:,} |" for c, v in violations.items()]
lines += ["", "## Per column completeness, as supplied", "",
          "| Column | Missing % | Min | Max | Mean | Zeros % |", "|---|---|---|---|---|---|"]
for _, r in quality_before.iterrows():
    lines.append(f"| `{r['column']}` | {r['missing_%']} | {r['min']} | {r['max']} | {r['mean']} | {r['zeros_%']} |")

(OUT/"data_quality_mastermeter1.md").write_text("\n".join(lines))
print(f"\nwrote {OUT/'data_quality_mastermeter1.md'}")

mastermeter1_5min.parquet             149.7 MB   (1,782,422 rows)
mastermeter1_hourly.parquet            17.2 MB   (148,537 rows)

wrote /Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting/Outputs/data_quality_mastermeter1.md
